# Metodología del Estudio por Simulación: GMM vs. K-Means

## 1. Formulación del Problema
Pregunta de Investigación

¿Cómo se degradan la precisión de partición geométrica y la estabilidad paramétrica de los centroides estimados por el algoritmo K-Means en comparación con un modelo de mezclas gaussianas (Gaussian Mixture Model, GMM) cuando se violan de forma progresiva y controlada los supuestos de esfericidad, homogeneidad de la varianza e independencia en los clústeres reales?
Objetivos del Estudio

    Evaluar la calidad de la partición: Cuantificar la coincidencia entre las etiquetas asignadas por los algoritmos y las etiquetas reales del mecanismo generador mediante el Índice Rand Ajustado (ARI) bajo diferentes configuraciones espaciales.

    Cuantificar la inestabilidad paramétrica mediante Bootstrap: Estimar la varianza empírica y los intervalos de confianza para las coordenadas espaciales de los centroides calculados por ambos algoritmos cuando la geometría de los datos diverge de los supuestos teóricos rígidos.

    Analizar el comportamiento en zonas de superposición difusa: Contrastar la sensibilidad de ambos métodos en escenarios con alta densidad compartida, analizando el impacto de una asignación dura (hard clustering) frente a una probabilística (soft clustering).

Hipótesis Previas

Se postula que en condiciones de esfericidad e igual varianza (escenario de control), K-Means y GMM exhibirán un rendimiento equivalente en términos de ARI. Sin embargo, ante la presencia de estructuras elípticas (anisotropía) o varianzas altamente desiguales (heterocedasticidad), K-Means sufrirá una degradación severa en su desempeño y una alta varianza en la localización de sus centroides calculada por Bootstrap, debido a su incapacidad de modelar la covarianza de los datos. Por el contrario, GMM mantendrá la robustez paramétrica y una degradación marginal de la métrica ARI en las mismas condiciones.

## 2. Modelización Conceptual

### Estructura del Problema y Factores
El problema se modela en un espacio bidimensional continuó ($D = 2$) donde coexisten grupos o clústeres latentes. Las variables predictoras son las coordenadas espaciales ($X_1, X_2$), y la variable respuesta es la etiqueta categórica del clúster ($Y$).

Los factores intervinientes se dividen en:

* **Factores fijos**: El número de observaciones totales ($N = 500$), el número real de clústeres ($K = 3$), la probabilidad a priori de pertenencia a cada grupo ($\pi_k = 1/3$) y la dimensión del espacio ($D = 2$).

* **Factores variables** (Escenarios): La matriz de covarianza de cada grupo ($\Sigma_k$) y la distancia euclídea entre los vectores de medias (centroides poblacionales, $\mu_k$).

### Supuestos de los Modelos respecto a la Relación de los Datos

    K-Means: Minimiza la inercia intraclúster basada en la distancia euclídea ordinaria. Asume implícitamente que los clústeres son esféricos (isotrópicos), poseen varianzas idénticas (homocedasticidad) y tamaños similares. Realiza una asignación dura donde las fronteras de decisión entre grupos son estrictamente lineales (diagramas de Voronoi).

    GMM: Maximiza la verosimilitud de que los datos provengan de una combinación lineal de componentes normales. Al estimar una matriz de covarianza completa para cada componente, asume que los grupos pueden ser elípticos (anisotrópicos), poseer diferentes orientaciones y varianzas desiguales. Realiza una asignación probabilística basada en el teorema de Bayes.

## 3. Definición del Mecanismo Generador de Datos (MGD)

El proceso generador de datos reales simulará una mixtura de distribuciones normales multivariadas en $\mathbb{R}^2$. Para garantizar la reproducibilidad absoluta de la simulación entre réplicas, se fijará una semilla aleatoria global (random_state) al inicio del flujo.

Para cada observación independiente $i$ (hasta completar $N = 500$):

1- Se sortea la pertenencia al clúster latente $Z_i \in \{1, 2, 3\}$ mediante una distribución categórica gobernada por las probabilidades a priori $\pi = [0.33, 0.33, 0.34]$.

2- Condicionado al valor obtenido de $Z_i = k$, se generan las coordenadas continuas vectoriales $X_i = [X_{i1}, X_{i2}]$ realizando un muestreo aleatorio de la correspondiente distribución Gaussiana multivariada:$$X_i | Z_i = k \sim \mathcal{N}(\mu_k, \Sigma_k)$$

3- El nivel de ruido y la dificultad del clustering se controlan directamente mediante el solapamiento algebraico de las distribuciones, modificando los autovalores de las matrices de covarianza $\Sigma_k$ en relación con la distancia entre los vectores de medias $\mu_k$.

## 4. Diseño de Escenarios

Se establecen cuatro escenarios fijos para evaluar el impacto de la violación de los supuestos geométricos:

* Escenario 1: Estructura Isotrópica Perfecta **(Control)**. Las matrices de covarianza son identidades multiplicadas por un escalar constante ($\Sigma_k = \sigma^2 I$). Los tres grupos son circulares, de igual tamaño y están muy distanciados.

* Escenario 2: Varianzas Altamente Desiguales **(Heterocedasticidad)**. Las matrices conservan la estructura diagonal (esfericidad), pero los escalares de varianza divergen fuertemente entre clústeres (ej. $\sigma_1^2 = 0.5$, $\sigma_2^2 = 2.0$, $\sigma_3^2 = 5.0$). Un grupo es muy compacto y los otros son dispersos.

* Escenario 3: Geometría Anisotrópica **(Estiramiento)**. Se introducen términos de covarianza no nulos fuera de la diagonal principal en las matrices $\Sigma_k$. Los clústeres se transforman en elipses alargadas con diferentes ángulos de inclinación en el plano. 

* Escenario 4: **Anisotropía Combinada con Superposición**. Se mantiene la estructura elíptica del Escenario 3, pero se reduce la distancia euclídea entre los vectores de medias $\mu_k$, forzando a los algoritmos a trabajar en una zona de alta incertidumbre espacial.

* Escenario 5: **Anisotropía Combinada con Superposición Más Agresiva**. Se mantiene la estructura elíptica del Escenario 4, pero se reduce la distancia euclídea aún más entre los vectores de medias $\mu_k$

## 5. Implementación y Registro de Métricas

El entorno de experimentación se construirá de forma modular en Python. Las funciones clave para generar datos, ejecutar Montecarlo y calcular Bootstrap se alojarán en scripts auxiliares (.py), mientras que la notebook (.ipynb) se reservará para la invocación, documentación narrativa y visualización.

Se implementarán dos flujos de evaluación independientes:

**Flujo A: Simulación de Montecarlo (Rendimiento General)**

Se ejecutarán $M = 200$ réplicas independientes para cada uno de los 4 escenarios. En cada réplica:

1- El MGD genera un nuevo dataset sintético balanceado ($N=500$).

2- Se ajustan un modelo KMeans(n_clusters=3) y un modelo GaussianMixture(n_components=3, covariance_type='full').

3- Se registra la métrica Adjusted Rand Index (ARI), la cual mide la similitud entre las etiquetas predichas por el algoritmo y las etiquetas reales del MGD, corregida por el azar (rango entre -1 y 1, donde 1 indica una partición perfecta).

**Flujo B: Remuestreo por Bootstrap (Estabilidad de Parámetros)**

Para un dataset fijo representativo de cada escenario, se realizarán $B = 500$ iteraciones de Bootstrap no paramétrico. En cada iteración:

1- Se genera una muestra con reemplazo de tamaño $N=500$ a partir del dataset base.

2- Se re-entrenan ambos modelos sobre la muestra Bootstrap.

3- Se registran las coordenadas espaciales $(X_1, X_2)$ de los centroides estimados para cada uno de los 3 clústeres (resolviendo previamente el problema de desalineación de etiquetas o label switching)

## 6. Experimentación

La experimentación automatizará el recorrido por los cuatro escenarios descritos. El número de réplicas de Montecarlo ($M = 200$) se establece para equilibrar el costo computacional con la estabilidad de los teoremas del límite central, asegurando que los estimadores de la media y la varianza del ARI converjan de forma prolija. De igual manera, las $B = 500$ submuestras de Bootstrap garantizan una densidad suficiente de centroides estimados para construir intervalos de confianza empíricos robustos y mapas de densidad precisos.

## 7. Análisis de Resultados y Conclusiones

Los datos recolectados en las etapas anteriores se procesarán estadísticamente y se presentarán en la notebook mediante dos núcleos visuales principales:

    Comparación de ARI (Montecarlo): Se estructurará un DataFrame con los resultados de las 200 réplicas por escenario y se construirán diagramas de caja (boxplots) agrupados por algoritmo. Esto permitirá visualizar de forma inmediata la pérdida de rendimiento (caída del ARI medio) y el incremento de la variabilidad del modelo rígido (K-Means) frente al flexible (GMM).

    Dispersión de Centroides (Bootstrap): Se graficarán diagramas de dispersión (scatter plots) donde cada iteración de Bootstrap proyectará un punto semitransparente con la ubicación del centroide estimado. En escenarios adversos, K-Means debería mostrar "nubes" de puntos de centroides dispersas y erráticas (reflejo de cortes inconsistentes en óvalos), mientras que GMM exhibirá agrupaciones concentradas y estables en torno a las medias poblacionales verdaderas.

    Conclusión Final: Se responderá formalmente a la pregunta de investigación, delimitando la regla práctica de cuándo el costo computacional adicional de un GMM se justifica plenamente frente a la simplicidad geométrica de un K-Means en entornos de producción con datos complejos.